# Plotting Tutorial

This tutorial walks through **every public plotting function** in `qdmpy.plotting`.
Each section demonstrates one function with real data so you can see what it
produces and how to call it.

| Section | Function | What it shows |
|---------|----------|---------------|
| 1 | -- | Data pipeline: load, fold, fit |
| 2 | `plot_measurement_images` | Light + laser optical images |
| 3 | `plot_odmr_spectra` | Raw ODMR spectra for one pixel |
| 4 | `plot_fluorescence_correction` | Before/after fluorescence correction |
| 5 | `plot_model_detection` | Median spectra with detected peaks |
| 6 | Folding diagnostics | Overview, search landscape, mean spectrum, pixel spectra |
| 7 | `plot_b111_map` / `plot_fit_result_field_map` | B111 remanent & induced maps |
| 8 | `plot_fit_result_parameter_map` | Single-parameter spatial maps |
| 9 | `plot_fit_result_overview` | Multi-panel parameter summary |
| 10 | `plot_magnetic_component` | Bx, By, Bz, Btotal from MagneticMap |
| 11 | `plot_qdm_display` | Comprehensive dashboard |
| 12 | Convenience methods | `result.plot()`, `result.show()`, `result.display()`, `m.display()` |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qdmpy.measurement import Measurement
import qdmpy.plotting as qplt

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

## 1. Data Pipeline

Load the MIL2_FOV1 dataset (4x binned for speed), fold, and fit.
These three objects -- `m`, `folded`, `result` -- are reused throughout.

In [ ]:
m = Measurement.from_folder(
    '../../tests/data/MIL2_FOV1',
    bin_factor=4,
    normalize=True,
)

folded = m.fold_odmr()
result = m.fit_folded_odmr(folded)

print(f'Measurement:  {m.odmr.processed_data.scan_dimensions}')
print(f'Folded shape: {tuple(folded.folded_spectrum.shape)}')
print(f'B111 remanent: mean={result.b111_remanent.mean():.1f} uT')

---

## 2. Measurement Images

`plot_measurement_images(measurement)` shows the light and laser
reference images side by side.

In [ ]:
qplt.plot_measurement_images(m)

---

## 3. ODMR Spectra

`plot_odmr_spectra(odmr_data, y, x)` shows the raw ODMR spectrum at
one pixel, with one subplot per (polarity, freq_range) combination.

In [ ]:
cy = m.odmr.processed_data.scan_dimensions[0] // 2
cx = m.odmr.processed_data.scan_dimensions[1] // 2

qplt.plot_odmr_spectra(m.odmr.processed_data, y=cy, x=cx)

---

## 4. Fluorescence Correction Preview

`plot_fluorescence_correction(odmr_data)` shows the original vs
corrected spectrum for a single pixel. The correction removes the
frequency-dependent fluorescence background.

In [ ]:
qplt.plot_fluorescence_correction(m.odmr.raw_data)

---

## 5. Model Detection

`plot_model_detection(spectra_4d, freq)` displays the median spectrum
per (polarity, freq_range) panel with auto-detected peaks marked.
Useful for verifying that the correct ESR model was chosen.

The function expects a 4D array `(n_pol, n_frange, n_pixel, n_freq)`,
so we flatten the spatial dimensions from the 5D processed data.

In [ ]:
data_5d = m.odmr.processed_data.data.values  # (pol, frange, y, x, freq)
n_pol, n_frange, ny, nx, n_freq = data_5d.shape
spectra_4d = data_5d.reshape(n_pol, n_frange, ny * nx, n_freq)

freq_ghz = m.odmr.processed_data.frequencies  # (n_frange, n_freq)

qplt.plot_model_detection(spectra_4d, freq=freq_ghz)

---

## 6. Spectral Folding Diagnostics

Four functions visualise different aspects of the folding result.

### 6a. Folding Overview

`plot_folding_overview(folded)` gives a 2x2 diagnostic grid: search
landscape per polarity, D_ZFS deviation map, and fold residual map.

In [ ]:
qplt.plot_folding_overview(folded)

### 6b. Search Landscape

`plot_folding_search_landscape(folded)` shows the brute-force D_ZFS
search residual vs candidate value. A clean single minimum means the
search converged well.

In [ ]:
qplt.plot_folding_search_landscape(folded)

### 6c. Mean Folded Spectrum

`plot_folding_mean_spectrum(folded)` shows the spatially-averaged
folded spectrum. The antisymmetric component (red) should be near zero.

In [ ]:
qplt.plot_folding_mean_spectrum(folded)

### 6d. Per-Pixel Spectra

`plot_folding_pixel_spectra(folded, x, y)` shows folded, unfolded, and
antisymmetric traces for individual pixels. Pass lists for multiple pixels.

In [ ]:
qplt.plot_folding_pixel_spectra(folded, x=cx, y=[0, cy, ny - 1])

---

## 7. B111 Field Maps

`plot_b111_map(result, component)` plots one B111 component as a
spatially-resolved map with a symmetric RdBu_r colourmap.

`plot_fit_result_field_map(result)` auto-detects the best field map
to show (B111 remanent for multi-range models, legacy B-field for
single-range).

In [ ]:
qplt.plot_b111_map(result, component='remanent')

In [ ]:
qplt.plot_b111_map(result, component='induced')

In [ ]:
qplt.plot_fit_result_field_map(result)

---

## 8. Parameter Maps

`plot_fit_result_parameter_map(result, param_name)` plots any single
fitted parameter as a spatial map. Common choices:

| Parameter | Description |
|-----------|-------------|
| `center` | Resonance centre frequency (GHz) |
| `chi2` | Fit quality |
| `width_0` | Linewidth |
| `contrast_0` | ODMR contrast (dip depth) |
| `offset` | Baseline offset |

In [ ]:
qplt.plot_fit_result_parameter_map(result, 'center')

In [ ]:
qplt.plot_fit_result_parameter_map(result, 'chi2')

---

## 9. Fit Result Overview

`plot_fit_result_overview(result)` shows a multi-panel summary of all
key parameters (B111, centre, linewidth, contrast, chi-squared) in one
figure.

In [ ]:
qplt.plot_fit_result_overview(result)

---

## 10. Magnetic Map Components

Access the 3D field reconstruction via `result.magnetic_map`, then
use `plot_magnetic_component(mag_map, component)` to display individual
components (Bx, By, Bz, Btotal).

In [ ]:
mag_map = result.magnetic_map
print(f'Components: b111, bx, by, bz, btotal')
print(f'NV axis: {mag_map.nv_axis}')

In [ ]:
qplt.plot_magnetic_component(mag_map, 'Bz')

In [ ]:
qplt.plot_magnetic_component(mag_map, 'Bx')

In [ ]:
qplt.plot_magnetic_component(mag_map, 'By')

In [ ]:
qplt.plot_magnetic_component(mag_map, 'Btotal')

---

## 11. Comprehensive Display

`plot_qdm_display(result, measurement=m)` is the full dashboard showing
B111 maps, parameter maps, optical images, and representative pixel
spectra with fitted model curves.

In [ ]:
qplt.plot_qdm_display(result, measurement=m)

---

## 12. Convenience Methods

The same plots are accessible as methods on the result and measurement
objects, so you don't need to import `qdmpy.plotting` directly.

| Method | Equivalent |
|--------|------------|
| `result.plot('center')` | `plot_fit_result_parameter_map(result, 'center')` |
| `result.plot('b111_remanent')` | `plot_b111_map(result, 'remanent')` |
| `result.show()` | `plot_fit_result_overview(result)` |
| `result.display(measurement=m)` | `plot_qdm_display(result, measurement=m)` |
| `m.plot()` | `plot_measurement_images(m)` |
| `m.display(result)` | `plot_qdm_display(result, measurement=m)` |

In [ ]:
result.plot('b111_remanent')

In [ ]:
result.show()

The next two cells produce **identical output** -- they are two spellings of
the same call. Use whichever reads more naturally in your workflow.

In [ ]:
result.display(measurement=m)

In [ ]:
m.display(result)